# Long lofi with ACE-Step (Colab T4)

Makes one continuous instrumental of any length, from a few minutes to several hours, using only [ACE-Step v1-3.5B](https://github.com/ace-step/ACE-Step).

1. ACE-Step generates the opening 2 minutes from your prompt.
2. ACE-Step's own **extend** task continues it. It gets the last 60 seconds as context and writes the next 2 minutes, and this repeats until the piece reaches `LENGTH`. Every continuation is composed from what came before, so the music carries on instead of being several tracks joined together.
3. The pieces are joined with a 1-second crossfade inside the overlap, where ACE-Step re-rendered the same audio. There's a fade-out at the end. Nothing else is added: no mastering or effects, and the audio is ACE-Step's own output.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Then run the cells from top to bottom.

## 1. Check the GPU

In [12]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))

name, memory.total [MiB]
Tesla T4, 15360 MiB
torch 2.11.0+cu128 | GPU: Tesla T4


## 2. Install ACE-Step

ACE-Step pins old package versions (e.g. `spacy==3.8.4`) that have no builds for Colab's Python 3.13. So it's installed with `--no-deps`, followed by the packages inference needs. `py3langid` stays at 0.3.0, because 0.4.0 removed a function ACE-Step calls.

pip will print `ERROR: pip's dependency resolver ... ace-step 0.2.0 requires X==...`. That's only a warning about those pins. If the version table prints at the end, the install worked.

In [13]:
!pip install -q --no-deps git+https://github.com/ace-step/ACE-Step.git
!pip install -q "diffusers>=0.33.0" "spacy>=3.8.7,<3.9" loguru pypinyin "py3langid==0.3.0" hangul-romanize num2words soundfile librosa peft accelerate
import importlib.metadata as md
for pkg in ["ace-step", "torch", "transformers", "diffusers", "py3langid"]:
    print(f"{pkg:13s}", md.version(pkg))

  Preparing metadata (setup.py) ... done
ace-step      0.2.0
torch         2.11.0+cu128
transformers  5.16.1
diffusers     0.40.0
py3langid     0.3.0


## 3. Hugging Face login (optional)

The weights are public, so a token isn't required, but it speeds up the 8 GB download. Add it as a Colab secret named `HF_TOKEN`: click 🔑 in the left sidebar, then turn on Notebook access.

In [14]:
import os
from huggingface_hub import login
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    token = None
    print("No HF_TOKEN secret; downloading anonymously.")
if token:
    os.environ["HF_TOKEN"] = token
    login(token=token, add_to_git_credential=False)
    print("Logged in to Hugging Face.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in to Hugging Face.


## 4. Load the model

- **Precision:** a T4 has no fast bfloat16, so the model runs in float16. If the output comes out silent or as noise, switch to `bfloat16` and run this cell again.
- **Decoding:** it uses ACE-Step's overlapped decode, its own low-memory setting for tracks over 48 seconds.
- **First run:** downloads about 8 GB.

In [15]:
import os, time
import torch
DTYPE = "float16"  #@param ["float16", "bfloat16"]
CHECKPOINT_DIR = "/content/ace_step_checkpoints"  #@param {type:"string"}
os.environ["ACE_PIPELINE_DTYPE"] = DTYPE   # ACE-Step reads this and it overrides its default

import numpy as np
import soundfile as sf
from acestep.pipeline_ace_step import ACEStepPipeline
from acestep.music_dcae.music_dcae_pipeline import MusicDCAE

# Newer torchaudio routes load() and save() through torchcodec, which Colab
# doesn't have, so ACE-Step reads and writes its WAVs with soundfile instead.
def _save_wav_file(self, target_wav, idx, save_path=None, sample_rate=48000, format="wav"):
    os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)
    sf.write(save_path, target_wav.float().cpu().numpy().T, sample_rate)
    return save_path

def _load_audio(self, audio_path):
    audio, sr = sf.read(audio_path, dtype="float32", always_2d=True)
    audio = torch.from_numpy(audio.T.copy())
    if audio.shape[0] == 1:
        audio = audio.repeat(2, 1)
    return audio, sr

ACEStepPipeline.save_wav_file = _save_wav_file
MusicDCAE.load_audio = _load_audio

t0 = time.time()
pipe = ACEStepPipeline(checkpoint_dir=CHECKPOINT_DIR, overlapped_decode=True)
pipe.load_checkpoint(pipe.checkpoint_dir)
pipe.loaded = True
print(f"Loaded in {time.time() - t0:.0f}s | dtype={pipe.dtype} | "
      f"VRAM {torch.cuda.memory_allocated() / 1e9:.1f} GB")

2026-09-23 13:50:04.793 | INFO     | acestep.pipeline_ace_step:get_checkpoint_path:178 - Download models from Hugging Face: ACE-Step/ACE-Step-v1-3.5B, cache to: /content/ace_step_checkpoints


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

There are modules in ACEStepTransformer2DModel that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.
/usr/local/lib/python3.13/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 3.81 MiB is free. Including non-PyTorch memory, this process has 14.56 GiB memory in use. Of the allocated memory 13.64 GiB is allocated by PyTorch, and 800.51 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

## 5. Settings

- **`PROMPT`**: ACE-Step tags describing the music. Keep `instrumental` in it.
- **`LENGTH`**: accepts `2h`, `45m`, `90s`, `1h 30m`, `1:30:00`, or a bare number of minutes.
- **`SEED`**: leave blank for a random one, so every run sounds different. Set a number to repeat a run.
- **`SAVE_TO_DRIVE`**: keeps the work on Google Drive, so a Colab disconnect doesn't lose it. To resume a run, put the folder name printed below into `RUN_NAME`, run steps 1–5 again, then step 6. It continues where it stopped.

In [ ]:
import json, math, random, re
from datetime import datetime
from pathlib import Path

PROMPT = "lofi hip hop, instrumental, chill, mellow, jazzy chords, rhodes piano, soft boom bap drums, warm bass, relaxed, 80 bpm"  #@param {type:"string"}
LENGTH = "10m"  #@param {type:"string"}
SEED = ""  #@param {type:"string"}
SAVE_TO_DRIVE = True  #@param {type:"boolean"}
RUN_NAME = ""  #@param {type:"string"}
INFER_STEPS = 60  #@param {type:"slider", min:30, max:100, step:5}
GUIDANCE_SCALE = 15.0  #@param {type:"number"}

FIRST_S = 120    # the opening, generated from the prompt alone
CONTEXT_S = 60   # how much of the music so far ACE-Step hears before continuing
EXTEND_S = 120   # how much each extension adds (context + extension stays under ACE-Step's 240 s)
XFADE_S = 1.0    # crossfade inside the overlap ACE-Step re-rendered
FADE_OUT_S = 8.0

def parse_length(text):
    """'2h', '45m', '90s', '1h 30m', '2m30s', '1:30:00', '45:00' -> seconds; bare number = minutes."""
    t = str(text).strip().lower().replace(" ", "")
    if ":" in t:
        secs = 0.0
        for part in t.split(":"):
            secs = secs * 60 + float(part)
        return secs
    if re.fullmatch(r"\d+(\.\d+)?", t):
        return float(t) * 60
    units = {"h": 3600, "hr": 3600, "hrs": 3600, "hour": 3600, "hours": 3600,
             "m": 60, "min": 60, "mins": 60, "minute": 60, "minutes": 60,
             "s": 1, "sec": 1, "secs": 1, "second": 1, "seconds": 1}
    pieces = re.findall(r"(\d+(?:\.\d+)?)([a-z]+)", t)
    if not pieces or "".join(n + u for n, u in pieces) != t or any(u not in units for _, u in pieces):
        raise ValueError(f"can't read length {text!r}; try '1h 30m', '45m', '90s' or '1:30:00'")
    return sum(float(n) * units[u] for n, u in pieces)

def fmt_length(secs):
    secs = int(round(secs))
    h, rem = divmod(secs, 3600)
    m, s = divmod(rem, 60)
    return " ".join(f"{v}{u}" for v, u in ((h, "h"), (m, "m"), (s, "s")) if v) or "0s"

target_s = parse_length(LENGTH)
if target_s < 20:
    raise ValueError("LENGTH must be at least 20 seconds")

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    runs_root = Path("/content/drive/MyDrive/ace_lofi_runs")
else:
    runs_root = Path("/content/ace_lofi_runs")
run = runs_root / (RUN_NAME.strip() or datetime.now().strftime("%Y%m%d-%H%M%S"))
(run / "segments").mkdir(parents=True, exist_ok=True)

state_path = run / "state.json"
if state_path.exists():
    state = json.loads(state_path.read_text())
    if state["prompt"] != PROMPT:
        print("Resuming with this run's original prompt, so the music stays consistent:\n  " + state["prompt"])
    state["target_s"] = target_s
else:
    state = {"prompt": PROMPT,
             "seed": int(SEED) if SEED.strip() else random.SystemRandom().randrange(2 ** 31),
             "target_s": target_s, "segments": []}
state_path.write_text(json.dumps(state, indent=2))

steps_needed = 1 + max(0, math.ceil((target_s - FIRST_S) / EXTEND_S))
print(f"Run folder: {run}\nRUN_NAME to resume: {run.name}\nSeed: {state['seed']}")
print(f"{fmt_length(target_s)} = 1 opening + {steps_needed - 1} extensions "
      f"(roughly {steps_needed * 1.5:.0f} min on a T4)")

## 6. Generate

Each step prints how long it took and an estimate of the time left. It's safe to stop and rerun, because finished segments are kept. If a take comes out silent or as noise, it's retried with the next seed.

In [ ]:
SR = 48000                         # ACE-Step writes 48 kHz
seg_dir, tmp = run / "segments", Path("/content/ace_tmp")
tmp.mkdir(exist_ok=True)
xf = int(XFADE_S * SR)
# ACE-Step's latent frame is 4096 samples at 44.1 kHz; a whole number of frames
# of context means its re-render of the context lines up with the original.
ctx_len = round(round(CONTEXT_S * 44100 / 4096) * 4096 / 44100 * SR)

COMMON = dict(format="wav", prompt=state["prompt"], lyrics="[instrumental]",
              infer_step=INFER_STEPS, guidance_scale=GUIDANCE_SCALE, scheduler_type="euler",
              cfg_type="apg", omega_scale=10.0, guidance_interval=0.5,
              guidance_interval_decay=0.0, min_guidance_scale=3.0, use_erg_tag=True,
              use_erg_lyric=True, use_erg_diffusion=True, oss_steps=None,
              guidance_scale_text=0.0, guidance_scale_lyric=0.0, batch_size=1)

def ace(seed, **task):
    out = tmp / "out.wav"
    pipe(**COMMON, manual_seeds=[seed], save_path=str(out), **task)
    audio, sr = sf.read(out, dtype="float32", always_2d=True)
    assert sr == SR, sr
    if not np.isfinite(audio).all() or np.abs(audio).max() < 1e-3:
        return None
    return audio

def context_end(ctx, out, search_s=0.1):
    """Where the re-rendered context ends in `out` (ACE-Step can shift it by a few samples)."""
    ref = ctx[-SR:].mean(axis=1)
    lo = max(0, ctx_len - SR - int(search_s * SR))
    hi = min(len(out), ctx_len + int(search_s * SR))
    corr = np.correlate(out[lo:hi].mean(axis=1), ref, mode="valid")
    return lo + int(np.argmax(corr)) + SR

def made_s():
    lens = [s["seconds"] for s in state["segments"]]
    return sum(lens) - XFADE_S * max(0, len(lens) - 1)

timings = []
print(f"Have {fmt_length(made_s())} of {fmt_length(state['target_s'])}.")
while made_s() < state["target_s"]:
    k = len(state["segments"])
    for attempt in range(3):
        seed = state["seed"] + 1000 * k + attempt
        t0 = time.time()
        if k == 0:
            audio = ace(seed, audio_duration=float(FIRST_S))
        else:
            prev, _ = sf.read(seg_dir / state["segments"][-1]["file"], dtype="float32", always_2d=True)
            ctx = prev[-ctx_len:]
            sf.write(tmp / "context.wav", ctx, SR)
            dur = len(ctx) / SR
            out = ace(seed, audio_duration=dur, task="extend", src_audio_path=str(tmp / "context.wav"),
                      repaint_start=0, repaint_end=dur + EXTEND_S,
                      retake_seeds=[seed], retake_variance=1.0)
            # keep XFADE_S of ACE-Step's re-rendered context in front, for the join
            audio = None if out is None else out[context_end(ctx, out) - xf:]
        if audio is None:
            print(f"  step {k + 1}: silent or noisy take, retrying with the next seed "
                  "(if this repeats, use DTYPE='bfloat16' in step 4)")
            continue
        name = f"seg_{k:04d}.wav"
        sf.write(seg_dir / name, audio, SR, subtype="FLOAT")
        state["segments"].append({"file": name, "seed": seed, "seconds": round(len(audio) / SR, 3)})
        state_path.write_text(json.dumps(state, indent=2))
        timings.append(time.time() - t0)
        left = max(0, math.ceil((state["target_s"] - made_s()) / EXTEND_S))
        print(f"  step {k + 1} ({'opening' if k == 0 else 'extension'}): {timings[-1]:.0f}s | "
              f"{fmt_length(made_s())} of {fmt_length(state['target_s'])} | "
              f"~{fmt_length(left * sum(timings) / len(timings))} left")
        break
    else:
        raise SystemExit("Three bad takes in a row. Set DTYPE='bfloat16' in step 4 and run steps 4-6 again.")
print(f"\nDone: {len(state['segments'])} segments, {fmt_length(made_s())}.")

## 7. Build the song

Joins the segments with the 1-second crossfades, trims to `LENGTH`, fades out the last 8 seconds and saves it in the run folder. The segments take about 1.7 GB of Drive per hour of music. Tick `DELETE_SEGMENTS` to remove them afterwards, but keep them if you might rebuild the song or make it longer. It then plays 10 seconds either side of the first three joins, so you can check they're seamless.

In [ ]:
from IPython.display import Audio, display
FORMAT = "mp3"  #@param ["mp3", "wav"]
DOWNLOAD = True  #@param {type:"boolean"}
DELETE_SEGMENTS = False  #@param {type:"boolean"}

target = int(state["target_s"] * SR)
fade = int(FADE_OUT_S * SR)
# Linear, not equal-power: both sides of the overlap are the same music (ACE-Step's
# re-render of the context), and equal-power fades would bump the level ~40% mid-join.
fade_in = np.linspace(0, 1, xf, dtype=np.float32)[:, None]
fade_out = 1.0 - fade_in
wav_path = run / "song.wav"
written, joins = 0, []

with sf.SoundFile(wav_path, "w", SR, 2, subtype="PCM_24") as f:
    def emit(block):
        global written
        block = block[: max(0, target - written)]
        if not len(block):
            return
        idx = np.arange(written, written + len(block))
        gain = np.clip((target - idx) / fade, 0, 1)[:, None]    # fade-out over the last FADE_OUT_S
        f.write(np.clip(block * gain, -1.0, 1.0))
        written += len(block)

    tail = None
    segs = state["segments"]
    for n, seg in enumerate(segs):
        a, _ = sf.read(seg_dir / seg["file"], dtype="float32", always_2d=True)
        if tail is not None:
            joins.append(written / SR)
            a = np.concatenate([tail * fade_out + a[:xf] * fade_in, a[xf:]])
        if n < len(segs) - 1:
            tail, a = a[-xf:], a[:-xf]
        emit(a)

print(f"{wav_path.name}: {fmt_length(written / SR)}, {len(joins)} joins")
for t in joins[:3]:
    start = max(0, int((t - 10) * SR))
    clip, _ = sf.read(wav_path, start=start, frames=20 * SR, dtype="float32")
    print(f"join at {int(t // 60)}:{int(t % 60):02d}")
    display(Audio(clip.T, rate=SR))

song = wav_path
if FORMAT == "mp3":
    import subprocess
    song = run / "song.mp3"
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(wav_path),
                    "-codec:a", "libmp3lame", "-b:a", "320k", str(song)], check=True)
    wav_path.unlink()           # about 1 GB per hour; the MP3 is what you keep
print("Saved:", song)
if DELETE_SEGMENTS:   # ~1.7 GB of segments per hour; keep them if you might rebuild or extend
    for seg in state["segments"]:
        (seg_dir / seg["file"]).unlink(missing_ok=True)
    print("Deleted the segments.")
if DOWNLOAD:
    from google.colab import files
    files.download(str(song))